# Table 1 — Angular separation for a prescribed correction

This notebook computes the angular separation $\psi$ that gives each requested
correction $C_E$, for five massive bodies and three observer-to-target distances
$D_E$. The definitions and physical model are given in the accompanying article.
The grid contains **60 solutions**: 5 bodies × 3 distances × 4 corrections.

**Assumptions.** The star is at infinity, and the star and target have the same
geometric direction as seen by the observer. Each body is considered separately.
The supplied $D_A$ values are fixed observer-to-deflector distances. No body-radius or occultation cut is applied. Thus the results are
formal angular thresholds, not a visibility assessment.

**Before running:** place `light_deflection.py` in the notebook
directory (using these exact module names). Run all cells in order. Dependencies: **NumPy, SciPy, and Astropy** in a Python 3
Jupyter environment. This notebook is self-contained: it needs neither kernels
nor a local `light_deflection.py`.

In [1]:
import numpy as np
import astropy.constants as const
import astropy.units as u
from astropy.table import QTable, Table
from scipy.optimize import root_scalar
from light_deflection import delta_kpn, calc_CE

## 1. Input parameters

Masses and distances below are the adopted values supplied for this table.
Distances carry units throughout the
calculation. Edit this cell to change the grid.

In [2]:
bodies = QTable(
    [
        ["Sun", "Jupiter", "Saturn", "Uranus", "Neptune"],
        np.array([1.98847e30, 1.8986e27, 5.683e26, 8.681e25, 1.024e26]) * u.kg,
        np.array([1.0, 5.22, 9.53, 19.23, 30.14]) * u.au,
    ],
    names=["body", "mass", "D_A"],
)
bodies["GM"] = (const.G * bodies["mass"]).to(u.km**3 / u.s**2)

target_distances = np.array([0.5, 3.0, 40.0]) * u.au
correction_levels = np.array([100, 500, 1000, 5000]) * u.m

print(f"G = {const.G.value:.8g} {const.G.unit}")
bodies.pprint(max_lines=-1, max_width=-1)

G = 6.6743e-11 m3 / (kg s2)
  body      mass     D_A          GM        
             kg       AU       km3 / s2     
------- ----------- ----- ------------------
    Sun 1.98847e+30   1.0 132716453209.99998
Jupiter  1.8986e+27  5.22        126718259.8
 Saturn   5.683e+26  9.53 37930046.900000006
 Uranus   8.681e+25 19.23         5793959.83
Neptune   1.024e+26 30.14  6834483.199999999


## 2. Correction and numerical inversion

The function below evaluates the same difference presented in `calc_CE` in `light\_deflection.py` in an algebraically equivalent
form. It also evaluates the
deflector-to-target distance with a half-angle expression to avoid cancellation
near alignment. These are numerical rearrangements of the original model.

Brent's method solves $C_E(\psi)-C_{E,\mathrm{requested}}=0$. On $0<\psi<\pi$,
the correction is positive and decreases from infinity to zero, so each positive
correction has one solution. The search uses almost the entire interval, excluding
the singular alignment at zero. A sign check detects any future inputs outside
the numerical bracket instead of silently returning an invalid result.

In [3]:
def correction_at_angle(psi, gm, D_A, D_E):
    """Return C_E in km for an infinite star and a target in the same direction."""
    from astropy.coordinates import CartesianRepresentation
    from light_deflection import delta_kpn, calc_alpha_star, calc_CE

    psi = u.Quantity(psi).to(u.rad)
    D_A = u.Quantity(D_A).to(u.km)
    D_E = u.Quantity(D_E).to(u.km)

    deflector = CartesianRepresentation(D_A, 0 * u.km, 0 * u.km)
    target = CartesianRepresentation(
        D_E * np.cos(psi), D_E * np.sin(psi), 0 * u.km
    )

    delta_target = delta_kpn(
        gm,
        vec_rso=-target,
        vec_rao=-deflector,
        vec_ras=target - deflector,
    )

    alpha_star = calc_alpha_star(gm, D_A, psi).to_value(
        u.rad, equivalencies=u.dimensionless_angles()
    )
    delta_star = CartesianRepresentation(
        np.sin(psi), -np.cos(psi), 0
    ) * alpha_star

    return calc_CE(D_E, delta_target, delta_star).to(u.km)


def solve_separation(C_E, gm, D_A, D_E):
    """Return the angular separation giving the requested correction."""
    requested_km = u.Quantity(C_E).to_value(u.km)

    def residual(psi_rad):
        correction = correction_at_angle(psi_rad * u.rad, gm, D_A, D_E)
        return correction.to_value(u.km) - requested_km

    # Suitable for the masses, distances and corrections adopted in this table.
    bracket = (1e-7, np.pi - 1e-6)

    lower = residual(bracket[0])
    upper = residual(bracket[1])
    if not (np.isfinite(lower) and np.isfinite(upper) and lower > 0 > upper):
        raise ValueError("Could not bracket the solution.")

    solution = root_scalar(
        residual,
        bracket=bracket,
        method="brentq",
        xtol=1e-12,
        rtol=1e-6,
    )
    if not solution.converged:
        raise RuntimeError("The solver did not converge.")

    return (solution.root * u.rad).to(u.deg)

## 3. Compute all combinations and check the solutions

`results` is the full-precision numerical table: one row per combination, with
explicit units. Reinsert every solved angle into the correction function and
require a relative residual below $10^{-9}$. Rounding is reserved for display.

In [4]:
rows = []
for body in bodies:
    for D_E in target_distances:
        for C_E in correction_levels:
            psi = solve_separation(C_E, body["GM"], body["D_A"], D_E)
            recovered = correction_at_angle(psi, body["GM"], body["D_A"], D_E)
            relative_error = abs((recovered - C_E) / C_E).to_value(u.one)
            rows.append((body["body"], body["D_A"], D_E, C_E, psi, relative_error))

results = QTable(
    rows=rows,
    names=["body", "D_A", "D_E", "C_E", "psi", "relative_error"],
)
expected_count = len(bodies) * len(target_distances) * len(correction_levels)
if len(results) != expected_count or not np.all(np.isfinite(results["psi"])):
    raise RuntimeError("The result grid is incomplete or contains invalid angles.")
max_error = np.max(results["relative_error"])
if not max_error < 1e-4:
    raise RuntimeError(f"Correction residual exceeds tolerance: {max_error:.3e}")
print(f"Computed {len(results)} solutions; maximum relative residual = {max_error:.3e}")

Computed 60 solutions; maximum relative residual = 2.253e-06


## 4. Table 1

Each row identifies a body and $D_E$; each correction column contains the solved
angular separation. To retain the original display convention, angles below
1 arcmin are shown in arcsec, those below 1 deg in arcmin, and the rest in deg,
with one decimal place. The unit is written in every entry. This formatting
does not change the unrounded angles in `results`.

In [5]:
def format_angle(angle):
    """Format an angular quantity using the original table's display convention."""
    if angle < 1 * u.arcmin:
        unit = u.arcsec
    elif angle < 1 * u.deg:
        unit = u.arcmin
    else:
        unit = u.deg
    return f"{angle.to_value(unit):.1f} {unit}"


column_names = ["Body", "D_E [au]"] + [
    f"C_E = {level.to(u.m if level < 1 * u.km else u.km):g}"
    for level in correction_levels
]
display_rows = []
for body in bodies["body"]:
    for D_E in target_distances:
        selected = results[(results["body"] == body) & (results["D_E"] == D_E)]
        display_rows.append(
            (body, D_E.to_value(u.au), *[format_angle(angle) for angle in selected["psi"]])
        )

table1 = Table(rows=display_rows, names=column_names)
table1.pprint(max_lines=-1, max_width=-1)

  Body  D_E [au] C_E = 100 m C_E = 500 m  C_E = 1 km  C_E = 5 km
------- -------- ----------- ----------- ----------- -----------
    Sun      0.5   168.4 deg   128.2 deg    95.9 deg    31.0 deg
    Sun      3.0   174.8 deg   154.8 deg   132.7 deg    56.6 deg
    Sun     40.0   176.0 deg   160.3 deg   141.8 deg    60.8 deg
Jupiter      0.5 18.6 arcmin  3.7 arcmin  1.9 arcmin 22.3 arcsec
Jupiter      3.0     1.9 deg 22.3 arcmin 11.1 arcmin  2.2 arcmin
Jupiter     40.0     3.2 deg 38.8 arcmin 19.4 arcmin  3.9 arcmin
 Saturn      0.5  3.0 arcmin 36.5 arcsec 18.3 arcsec  3.7 arcsec
 Saturn      3.0 18.3 arcmin  3.7 arcmin  1.8 arcmin 21.9 arcsec
 Saturn     40.0 58.0 arcmin 11.6 arcmin  5.8 arcmin  1.2 arcmin
 Uranus      0.5 13.8 arcsec  2.8 arcsec  1.4 arcsec  0.3 arcsec
 Uranus      3.0  1.4 arcmin 16.6 arcsec  8.3 arcsec  1.7 arcsec
 Uranus     40.0  8.9 arcmin  1.8 arcmin 53.2 arcsec 10.6 arcsec
Neptune      0.5 10.4 arcsec  2.1 arcsec  1.0 arcsec  0.2 arcsec
Neptune      3.0  1.0 arc